### API Spotify

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.oauth2 import SpotifyOAuth
import passwords

In [ ]:
# Set up Spotify API credentials
client_id = passwords.id
client_secret = passwords.secret
client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

#### Obtención de información de las canciones de una playlist

In [ ]:
# Obtener la playlist 'Top 50 - Global' (ID oficial de Spotify)
playlist_id = passwords.playlist ## Hay que pasarle un ID de la playlist(también de las canciones). Aparece en el enlace
top_tracks = sp.playlist_items(playlist_id, limit=10)

# 10 canciones más populares
for idx, item in enumerate(top_tracks['items']):
    track = item['track']
    name = track['name']
    artists = ", ".join(artist['name'] for artist in track['artists'])
    popularity = track['popularity']
    duration_ms = track['m'] // 1000  # Convertir a segundos
    spotify_url = track['external_urls']['spotify']

    print(f"{idx + 1}. {name} by {artists}")
    print(f"   Popularity: {popularity} | Duration: {duration_ms} seconds")
    print(f"   Spotify URL: {spotify_url}\n")

##### Géneros

In [ ]:
# Obtener los géneros del primer artista de cada canción
for idx, item in enumerate(top_tracks['items']):
    track = item['track']
    artist_id = track['artists'][0]['id']  # ID del artista principal
    artist_info = sp.artist(artist_id)  # Obtener información del artista

    print(f"{idx + 1}. {track['name']} by {track['artists'][0]['name']}")
    print(f"   Genres: {', '.join(artist_info['genres'])}\n")

##### Info sobre la canción (tempo, popularidad...)

In [ ]:
# Obtener canciones de la playlist 'Top 50 - Global'
playlist_id = passwords.playlist  # ID de 'Top 50 - Global'
top_tracks = sp.playlist_items(playlist_id, limit=10)

# Recorrer las canciones y obtener información detallada
for idx, item in enumerate(top_tracks['items']):
    track = item['track']
    print(f"{idx + 1}. {track['name']} by {', '.join(artist['name'] for artist in track['artists'])}")
    print(f"   Spotify URL: {track['external_urls']['spotify']}")
    print(f"   Album: {track['album']['name']}")
    print(f"   Release Date: {track['album']['release_date']}")
    print(f"   Popularity: {track['popularity']}")
    
    # Audio Features
    audio_features = sp.audio_features(track['id'])[0]
    print(f"   Tempo: {audio_features['tempo']} BPM")
    print(f"   Energy: {audio_features['energy']}")
    print(f"   Danceability: {audio_features['danceability']}")
    print(f"   Valence: {audio_features['valence']}\n")

#### Búsqueda de canciones en concreto

Esto es lo que nos interesaría más, pasarle las canciones de la lista billboard para acceder a toda su info

In [ ]:
# Buscar una canción por nombre
query = "All I Want For Christmas Is You Mariah Carey"  # Nombre de la canción y artista
results = sp.search(q=query, type='track', limit=1)

# Extraer información
if results['tracks']['items']:
    track = results['tracks']['items'][0]
    print(f"Name: {track['name']}")
    print(f"Artist: {', '.join(artist['name'] for artist in track['artists'])}")
    print(f"Track ID: {track['id']}")
    print(f"Spotify URL: {track['external_urls']['spotify']}")
else:
    print("No se encontraron resultados.")

In [ ]:
# Buscar una canción por nombre
query = "nomadas"  # Nombre de la canción y artista
results = sp.search(q=query, type='track', limit=1)

# Extraer información
if results['tracks']['items']:
    track = results['tracks']['items'][0]
    print(f"Name: {track['name']}")
    print(f"Artist: {', '.join(artist['name'] for artist in track['artists'])}")
    print(f"Track ID: {track['id']}")
    print(f"Spotify URL: {track['external_urls']['spotify']}")
else:
    print("No se encontraron resultados.")

### Prueba de sacar datos de Spotify con datos de billboard


In [ ]:
# Set up Spotify API credentials
client_id = passwords.id
client_secret = passwords.secret
client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)

In [ ]:
sp = spotipy.Spotify(auth_manager = SpotifyOAuth(
    client_id = client_id,
    client_secret = client_secret,
    redirect_uri = "http://localhost:8888/callback",
    scope = "user-top-read"
))

In [ ]:
df_billboard = pd.read_csv("billboard100.csv")

In [ ]:
# Función para buscar canción y obtener datos necesarios
def get_song_data(title):
    try:
        # Búsqueda de la canción y de los parámetros que nos interesan
        results = sp.search(q=title, type='track', limit=1)
        if results['tracks']['items']:
            track = results['tracks']['items'][0]
            track_id = track['id']
            popularity = track['popularity']
            audio_features = sp.audio_features(track_id)[0]
            danceability = audio_features['danceability']
            tempo = audio_features['tempo']
            
            return {
                'popularity': popularity,
                'danceability': danceability,
                'tempo': tempo
            }
        else:
            return {'popularity': None, 'danceability': None, 'tempo': None}
    except Exception as e:
        print(f"Error fetching data for {title}: {e}")
        return {'popularity': None, 'danceability': None, 'tempo': None}

In [ ]:
df_billboard_100 = df_billboard.head(100)
df_billboard_100.loc[:, ['popularity', 'danceability', 'tempo']] = df_billboard_100['Title'].apply(get_song_data).apply(pd.Series)

df_billboard_100